# Phase 15 — Unlabeled Target Calibration and Prior-Shift Correction

**NetShield goal:** improve UNSW-NB15 → CICIDS2017 cross-dataset intrusion-detection performance.

Phase 14 showed that the expanded semantic representation + CORAL can improve **target ranking** (especially PR-AUC / ROC-AUC) while producing too many false positives at the source-derived decision threshold.

This phase therefore targets the next bottleneck: **decision calibration**.

### Main question
Can we improve cross-dataset F1 by estimating the CICIDS target class prior from **unlabeled target prediction scores**, then correcting the source model's posterior probabilities?

### Methods
1. Expanded semantic representation.
2. CORAL using unlabeled CICIDS adaptation data.
3. Random Forest with class weighting.
4. Source-only threshold baseline.
5. Source-validation probability calibration (Platt/logistic calibration).
6. Unlabeled target-prior estimation using the **Saerens-Latinne-Decaestecker EM procedure**.
7. Prior-corrected target probabilities and a decision threshold derived without CICIDS labels.

### Leakage rule
CICIDS labels are **never used to fit the model, CORAL, calibrator, prior estimator, or threshold**. They are read only after all decisions are frozen, for the final evaluation.

For apples-to-apples comparison, the CICIDS evaluation split reproduces the fixed stratified Phase 12 split (`random_state=42`). The labels are used only to construct that fixed benchmark partition, not to tune the method.


In [ ]:
from pathlib import Path
import gc
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix, brier_score_loss
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Robust project-root discovery
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for root in candidates:
    if (root / "data" / "raw").exists() or (root / "outputs").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

RAW_UNSW = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
RAW_CICIDS = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"

TABLE_DIR = PROJECT_ROOT / "outputs" / "phase15_target_calibration" / "tables"
PLOT_DIR = PROJECT_ROOT / "outputs" / "phase15_target_calibration" / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_UNSW:", RAW_UNSW)
print("RAW_CICIDS:", RAW_CICIDS)


In [ ]:
# -----------------------------
# UNSW-NB15 common representation
# -----------------------------
def load_unsw_common():
    all_csvs = list(RAW_UNSW.rglob("*.csv"))
    train_files = [p for p in all_csvs if "training" in p.name.lower()]
    test_files = [p for p in all_csvs if "testing" in p.name.lower()]

    if not train_files or not test_files:
        raise FileNotFoundError(
            f"Could not find UNSW training/testing CSVs under {RAW_UNSW}"
        )

    train_file = sorted(train_files)[0]
    test_file = sorted(test_files)[0]

    print("UNSW training:", train_file)
    print("UNSW testing :", test_file)

    tr = pd.read_csv(train_file, low_memory=False)
    te = pd.read_csv(test_file, low_memory=False)
    df = pd.concat([tr, te], ignore_index=True)

    y = pd.to_numeric(df["label"], errors="coerce").astype("Int8")

    dur = pd.to_numeric(df["dur"], errors="coerce")
    fp = pd.to_numeric(df["spkts"], errors="coerce")
    bp = pd.to_numeric(df["dpkts"], errors="coerce")
    fb = pd.to_numeric(df["sbytes"], errors="coerce")
    bb = pd.to_numeric(df["dbytes"], errors="coerce")

    total_p = fp + bp
    total_b = fb + bb
    safe_dur = dur.replace(0, np.nan)
    safe_total_p = total_p.replace(0, np.nan)

    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = dur
    x["fwd_packets"] = fp
    x["bwd_packets"] = bp
    x["fwd_bytes"] = fb
    x["bwd_bytes"] = bb
    x["total_packets"] = total_p
    x["total_bytes"] = total_b
    x["packet_rate"] = total_p / safe_dur
    x["fwd_packet_rate"] = fp / safe_dur
    x["bwd_packet_rate"] = bp / safe_dur
    x["avg_packet_size"] = total_b / safe_total_p

    return x, y


# -----------------------------
# CICIDS2017 common representation
# -----------------------------
def load_cicids_common():
    needed = [
        "Flow Duration",
        "Total Fwd Packets",
        "Total Backward Packets",
        "Total Length of Fwd Packets",
        "Total Length of Bwd Packets",
        "Flow Packets/s",
        "Fwd Packets/s",
        "Bwd Packets/s",
        "Average Packet Size",
        "Label",
    ]

    frames = []

    for p in sorted(RAW_CICIDS.rglob("*.csv")):
        h = pd.read_csv(p, nrows=0)
        lookup = {
            str(c).strip().replace("﻿", "").lower(): c
            for c in h.columns
        }

        sel = {
            c: lookup[c.lower()]
            for c in needed
            if c.lower() in lookup
        }

        if "Label" not in sel:
            print("Skipping:", p.name)
            continue

        chunks = []
        for chunk in pd.read_csv(
            p,
            usecols=list(sel.values()),
            chunksize=100_000,
            low_memory=True
        ):
            chunk = chunk.rename(columns={v: k for k, v in sel.items()})

            for c in needed:
                if c != "Label" and c in chunk.columns:
                    chunk[c] = pd.to_numeric(chunk[c], errors="coerce")

            chunk["Label"] = chunk["Label"].astype(str).str.strip()
            chunks.append(chunk)

        file_df = pd.concat(chunks, ignore_index=True)
        frames.append(file_df)
        print("Loaded:", p.name, len(file_df))

        del chunks, file_df
        gc.collect()

    if not frames:
        raise ValueError("No CICIDS CSV with Label found.")

    df = pd.concat(frames, ignore_index=True)

    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = df["Flow Duration"] / 1e6
    x["fwd_packets"] = df["Total Fwd Packets"]
    x["bwd_packets"] = df["Total Backward Packets"]
    x["fwd_bytes"] = df["Total Length of Fwd Packets"]
    x["bwd_bytes"] = df["Total Length of Bwd Packets"]
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = df["Flow Packets/s"]
    x["fwd_packet_rate"] = df["Fwd Packets/s"]
    x["bwd_packet_rate"] = df["Bwd Packets/s"]
    x["avg_packet_size"] = df["Average Packet Size"]

    y = (
        df["Label"].astype(str).str.strip().str.upper()
        .ne("BENIGN").astype("int8")
    )

    return x, y


X_unsw_11, y_unsw = load_unsw_common()
X_cic_11, y_cic = load_cicids_common()

print("UNSW:", X_unsw_11.shape)
print("CICIDS:", X_cic_11.shape)


In [ ]:
COMMON_11 = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size"
]

def build_expanded_semantic(X):
    x = X[COMMON_11].copy()
    eps = 1e-9

    x["fwd_packet_fraction"] = x["fwd_packets"] / (x["total_packets"] + eps)
    x["bwd_packet_fraction"] = x["bwd_packets"] / (x["total_packets"] + eps)
    x["fwd_byte_fraction"] = x["fwd_bytes"] / (x["total_bytes"] + eps)
    x["bwd_byte_fraction"] = x["bwd_bytes"] / (x["total_bytes"] + eps)

    x["fwd_bwd_packet_ratio"] = x["fwd_packets"] / (x["bwd_packets"] + eps)
    x["fwd_bwd_byte_ratio"] = x["fwd_bytes"] / (x["bwd_bytes"] + eps)

    x["fwd_bytes_per_packet"] = x["fwd_bytes"] / (x["fwd_packets"] + eps)
    x["bwd_bytes_per_packet"] = x["bwd_bytes"] / (x["bwd_packets"] + eps)
    x["bytes_per_packet"] = x["total_bytes"] / (x["total_packets"] + eps)

    safe_duration = x["flow_duration_sec"].replace(0, np.nan)
    x["byte_rate"] = x["total_bytes"] / safe_duration
    x["fwd_byte_rate"] = x["fwd_bytes"] / safe_duration
    x["bwd_byte_rate"] = x["bwd_bytes"] / safe_duration

    for c in [
        "flow_duration_sec", "fwd_packets", "bwd_packets",
        "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
        "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
        "avg_packet_size", "byte_rate", "fwd_byte_rate", "bwd_byte_rate"
    ]:
        x[f"log1p_{c}"] = np.log1p(x[c].clip(lower=0))

    return x.replace([np.inf, -np.inf], np.nan)

X_unsw_exp = build_expanded_semantic(X_unsw_11)
X_cic_exp = build_expanded_semantic(X_cic_11)

# Conservative numerical cleaning only.
def clean_numeric(X):
    X = X.copy().replace([np.inf, -np.inf], np.nan)
    f32_max = np.finfo(np.float32).max
    return X.mask(X.abs() > f32_max)

X_unsw_exp = clean_numeric(X_unsw_exp)
X_cic_exp = clean_numeric(X_cic_exp)

print("Expanded feature count:", X_unsw_exp.shape[1])


## 1. Reproduce the Phase 12 benchmark split

UNSW is split into source-fit/source-validation data exactly as before.

CICIDS is split into a **fixed 80/20 stratified evaluation split** with `random_state=42`, matching Phase 12. Up to 300,000 rows are sampled from the remaining pool for unlabeled CORAL adaptation.

The CICIDS labels are not passed into the model/adaptation/calibration procedures. They are retained only so the final frozen evaluation can be scored.


In [ ]:
src_fit_idx, src_val_idx = train_test_split(
    np.arange(len(X_unsw_exp)),
    test_size=0.20,
    stratify=y_unsw,
    random_state=RANDOM_STATE
)

cic_pool_idx, cic_eval_idx = train_test_split(
    np.arange(len(X_cic_exp)),
    test_size=0.20,
    stratify=y_cic,
    random_state=RANDOM_STATE
)

rng = np.random.default_rng(RANDOM_STATE)
ADAPT_MAX = min(300_000, len(cic_pool_idx))
cic_adapt_idx = rng.choice(
    cic_pool_idx, size=ADAPT_MAX, replace=False
)

print("UNSW source-fit:", len(src_fit_idx))
print("UNSW source-validation:", len(src_val_idx))
print("CICIDS unlabeled adaptation sample:", len(cic_adapt_idx))
print("CICIDS frozen evaluation:", len(cic_eval_idx))
print("Frozen CICIDS attack rate:", float(y_cic.iloc[cic_eval_idx].mean()))


In [ ]:
# Imputation is fitted only on UNSW source-fit data.
imp = SimpleImputer(strategy="median")

Xsrc_fit = imp.fit_transform(X_unsw_exp.iloc[src_fit_idx]).astype("float64")
Xsrc_val = imp.transform(X_unsw_exp.iloc[src_val_idx]).astype("float64")
Xt_adapt = imp.transform(X_cic_exp.iloc[cic_adapt_idx]).astype("float64")
Xt_eval = imp.transform(X_cic_exp.iloc[cic_eval_idx]).astype("float64")

y_src_fit = y_unsw.iloc[src_fit_idx].to_numpy()
y_src_val = y_unsw.iloc[src_val_idx].to_numpy()
y_tgt_eval = y_cic.iloc[cic_eval_idx].to_numpy()

print(Xsrc_fit.shape, Xsrc_val.shape, Xt_adapt.shape, Xt_eval.shape)


In [ ]:
# Corrected CORAL: source is aligned to the unlabeled target covariance.
def symmetric_inv_sqrt(C, eps=1e-6):
    vals, vecs = np.linalg.eigh(C)
    vals = np.clip(vals, eps, None)
    return (vecs * (1.0 / np.sqrt(vals))) @ vecs.T

def symmetric_sqrt(C, eps=1e-6):
    vals, vecs = np.linalg.eigh(C)
    vals = np.clip(vals, eps, None)
    return (vecs * np.sqrt(vals)) @ vecs.T

def fit_coral(Xs, Xt):
    source_mean = Xs.mean(axis=0)
    target_mean = Xt.mean(axis=0)

    Zs = Xs - source_mean
    Zt = Xt - target_mean

    Cs = np.cov(Zs, rowvar=False)
    Ct = np.cov(Zt, rowvar=False)

    A = symmetric_inv_sqrt(Cs) @ symmetric_sqrt(Ct)

    return {
        "source_mean": source_mean,
        "target_mean": target_mean,
        "A": A,
        "Cs": Cs,
        "Ct": Ct,
    }

def apply_coral_source(X, coral):
    return (X - coral["source_mean"]) @ coral["A"] + coral["target_mean"]

def covariance_distance(A, B):
    return float(np.linalg.norm(
        np.cov(A, rowvar=False) - np.cov(B, rowvar=False),
        ord="fro"
    ))

# Standardize using source-fit statistics.
scaler = StandardScaler()
Zs_fit = scaler.fit_transform(Xsrc_fit)
Zs_val = scaler.transform(Xsrc_val)
Zt_adapt = scaler.transform(Xt_adapt)
Zt_eval = scaler.transform(Xt_eval)

coral = fit_coral(Zs_fit, Zt_adapt)

Xsrc_coral = apply_coral_source(Zs_fit, coral)
Xval_coral = apply_coral_source(Zs_val, coral)

print("Covariance distance:",
      covariance_distance(Zs_fit, Zt_adapt),
      "->",
      covariance_distance(Xsrc_coral, Zt_adapt))


In [ ]:
RF_PARAMS = dict(
    n_estimators=250,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)

rf = RandomForestClassifier(
    **RF_PARAMS,
    class_weight="balanced"
)

rf.fit(Xsrc_coral, y_src_fit)

source_val_raw = rf.predict_proba(Xval_coral)[:, 1]
target_raw = rf.predict_proba(Zt_eval)[:, 1]

source_prior = float(y_src_fit.mean())
print("Source-fit attack prior:", source_prior)
print("Source validation score range:", source_val_raw.min(), source_val_raw.max())
print("Target score range:", target_raw.min(), target_raw.max())


## 2. Baseline: source-only decision threshold

The threshold is selected using **UNSW source-validation labels only**.

This is the clean baseline against which all target-calibration methods are compared.


In [ ]:
def select_threshold(y_true, proba, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 181)
    scores = [
        f1_score(y_true, (proba >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])

def evaluate_binary(y_true, proba, threshold, name):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    return {
        "method": name,
        "threshold": float(threshold),
        "f1": f1_score(y_true, pred, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_attack_rate": float(pred.mean())
    }

source_threshold = select_threshold(y_src_val, source_val_raw)
print("Source-only threshold:", source_threshold)


## 3. Source-validation probability calibration

Random Forest probabilities are not guaranteed to be calibrated.

We fit a simple logistic (Platt-style) calibrator using only the source-validation scores and labels. The calibrator is then frozen before looking at CICIDS labels.

This does not create a new classifier; it only maps model scores to better-behaved source-domain probabilities.


In [ ]:
# Logistic calibration on source-validation scores.
# Clip probabilities before logit transformation.
def logit(p):
    p = np.clip(np.asarray(p), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

calibrator = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    random_state=RANDOM_STATE,
    max_iter=1000
)

calibrator.fit(
    logit(source_val_raw).reshape(-1, 1),
    y_src_val
)

source_val_cal = calibrator.predict_proba(
    logit(source_val_raw).reshape(-1, 1)
)[:, 1]

target_cal = calibrator.predict_proba(
    logit(target_raw).reshape(-1, 1)
)[:, 1]

calibrated_source_threshold = select_threshold(
    y_src_val, source_val_cal
)

print("Calibrated source threshold:", calibrated_source_threshold)
print("Source Brier raw:", brier_score_loss(y_src_val, source_val_raw))
print("Source Brier calibrated:", brier_score_loss(y_src_val, source_val_cal))


## 4. Unlabeled target-prior estimation with EM

The source model was trained on a much higher attack prior than CICIDS. A model's posterior probabilities can therefore be systematically miscalibrated after transfer.

We estimate the target attack prior from the **unlabeled CICIDS target scores** using an EM procedure under the prior-shift assumption:

- class-conditional feature/score behavior is approximately stable;
- the main change is the class prior.

The algorithm iteratively:
1. estimates target posterior probabilities using a candidate target prior;
2. updates the target attack prior using the mean posterior;
3. repeats until convergence.

No CICIDS labels are used.


In [ ]:
def estimate_target_prior_em(
    source_calibrated_probs,
    source_labels,
    target_calibrated_probs,
    max_iter=200,
    tol=1e-7,
    min_prior=1e-4,
    max_prior=1-1e-4
):
    # Estimate source prior from the source validation set.
    pi_s = float(np.mean(source_labels))

    # We need class-conditional score densities.
    # Use histograms on source-validation calibrated probabilities.
    bins = np.linspace(0.0, 1.0, 201)
    eps = 1e-12

    p0, _ = np.histogram(
        source_calibrated_probs[source_labels == 0],
        bins=bins, density=True
    )
    p1, _ = np.histogram(
        source_calibrated_probs[source_labels == 1],
        bins=bins, density=True
    )

    # Normalize to probability masses rather than raw density heights.
    p0 = p0 + eps
    p1 = p1 + eps
    p0 = p0 / p0.sum()
    p1 = p1 / p1.sum()

    # Source prior implied by validation data.
    pi_s = np.clip(pi_s, min_prior, max_prior)
    pi_t = pi_s

    target_bins = np.clip(
        np.digitize(target_calibrated_probs, bins) - 1,
        0, len(p0) - 1
    )

    for iteration in range(max_iter):
        f0 = p0[target_bins]
        f1 = p1[target_bins]

        numerator = pi_t * f1
        denominator = numerator + (1 - pi_t) * f0
        posterior_t = numerator / np.maximum(denominator, eps)

        new_pi_t = float(np.clip(
            posterior_t.mean(), min_prior, max_prior
        ))

        if abs(new_pi_t - pi_t) < tol:
            pi_t = new_pi_t
            break

        pi_t = new_pi_t

    return pi_t, iteration + 1, posterior_t


estimated_target_prior, em_iterations, target_em_posterior = (
    estimate_target_prior_em(
        source_val_cal,
        y_src_val,
        target_cal
    )
)

print("Estimated target attack prior:", estimated_target_prior)
print("EM iterations:", em_iterations)
print("Observed target score mean:", float(target_cal.mean()))


## 5. Prior correction

Given source prior `πs` and estimated target prior `πt`, correct the calibrated source-model odds:

`odds_target = odds_source × [πt/(1−πt)] / [πs/(1−πs)]`

This produces target-domain probabilities without using target labels.

We evaluate two versions:

- **Prior-corrected probabilities + 0.50 decision rule**
- **Prior-corrected probabilities + threshold selected on source validation**

The second threshold is still source-only; it is not tuned on CICIDS.


In [ ]:
def prior_correct_probabilities(p_source, source_prior, target_prior):
    p_source = np.clip(np.asarray(p_source), 1e-7, 1 - 1e-7)

    source_odds = p_source / (1 - p_source)
    target_odds = (
        source_odds
        * (target_prior / (1 - target_prior))
        / (source_prior / (1 - source_prior))
    )

    return target_odds / (1 + target_odds)

target_prior_corrected = prior_correct_probabilities(
    target_cal,
    source_prior=float(y_src_val.mean()),
    target_prior=estimated_target_prior
)

source_val_prior_corrected = prior_correct_probabilities(
    source_val_cal,
    source_prior=float(y_src_val.mean()),
    target_prior=estimated_target_prior
)

prior_threshold = select_threshold(
    y_src_val,
    source_val_prior_corrected
)

print("Prior-corrected source threshold:", prior_threshold)
print("Corrected target probability range:",
      target_prior_corrected.min(),
      target_prior_corrected.max())


## 6. Optional unlabeled threshold diagnostic

The following diagnostics are **not used as final tuning**.

They show how the target score distribution changes as the decision threshold moves. This is useful for understanding why calibration helps or fails, but the final reported method must use only source-derived parameters plus unlabeled target adaptation.


In [ ]:
# Label-free threshold candidates based only on predicted target attack rate.
# These are diagnostics, not selected using target labels.
target_attack_rate_before = float((target_cal >= calibrated_source_threshold).mean())
target_attack_rate_after = float((target_prior_corrected >= 0.5).mean())

diagnostic_df = pd.DataFrame({
    "quantity": [
        "source_attack_prior",
        "estimated_target_attack_prior",
        "target_attack_rate_at_source_threshold",
        "target_attack_rate_after_prior_correction_0.5"
    ],
    "value": [
        float(y_src_val.mean()),
        float(estimated_target_prior),
        target_attack_rate_before,
        target_attack_rate_after
    ]
})

display(diagnostic_df)
diagnostic_df.to_csv(
    TABLE_DIR / "01_unlabeled_prior_diagnostic.csv",
    index=False
)


In [ ]:
# Final frozen evaluation.
# CICIDS labels enter the computation only here.
results = []

results.append(
    evaluate_binary(
        y_tgt_eval,
        target_raw,
        source_threshold,
        "expanded + CORAL + balanced | source threshold"
    )
)

results.append(
    evaluate_binary(
        y_tgt_eval,
        target_cal,
        calibrated_source_threshold,
        "expanded + CORAL + balanced | calibrated source threshold"
    )
)

results.append(
    evaluate_binary(
        y_tgt_eval,
        target_prior_corrected,
        0.50,
        "expanded + CORAL + balanced | EM prior correction"
    )
)

results.append(
    evaluate_binary(
        y_tgt_eval,
        target_prior_corrected,
        prior_threshold,
        "expanded + CORAL + balanced | EM prior correction + source threshold"
    )
)

results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "method", "f1", "macro_f1", "precision", "recall",
            "pr_auc", "roc_auc", "fpr", "threshold",
            "predicted_attack_rate", "tn", "fp", "fn", "tp"
        ]
    ].sort_values("f1", ascending=False)
)

results_df.to_csv(
    TABLE_DIR / "02_phase15_final_results.csv",
    index=False
)


In [ ]:
# Compare against the established Phase 12 benchmark.
PHASE12_BENCHMARK_F1 = 0.3385

best_row = results_df.sort_values(
    ["f1", "pr_auc", "roc_auc"],
    ascending=False
).iloc[0]

summary = {
    "phase12_benchmark_f1": PHASE12_BENCHMARK_F1,
    "phase15_best_f1": float(best_row["f1"]),
    "phase15_best_method": best_row["method"],
    "f1_gain_over_phase12": float(best_row["f1"] - PHASE12_BENCHMARK_F1),
    "estimated_target_attack_prior": float(estimated_target_prior),
    "source_attack_prior": float(y_src_val.mean()),
    "em_iterations": int(em_iterations),
    "evaluation_n": int(len(cic_eval_idx)),
    "adaptation_n": int(len(cic_adapt_idx)),
    "leakage_rule": (
        "CICIDS labels used only for final frozen evaluation; "
        "no CICIDS labels used to fit CORAL, calibrator, prior estimator, "
        "or decision threshold."
    )
}

print(json.dumps(summary, indent=2))

with open(TABLE_DIR / "03_phase15_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nInterpretation:")
if best_row["f1"] > PHASE12_BENCHMARK_F1:
    print("SUCCESS: Phase 15 beats the Phase 12 F1 benchmark.")
else:
    print("Phase 15 does not yet beat the Phase 12 F1 benchmark.")
    print("Do not force a positive conclusion; use the ranking/calibration results to decide the next adaptation method.")


## 7. How to interpret the outcome

### Success criterion
The primary success criterion is:

**Phase 15 F1 > 0.3385 on the harmonized Phase 12 CICIDS evaluation split**, while also checking PR-AUC, Macro-F1, Recall, and FPR.

A useful improvement should not simply increase recall by predicting almost everything as attack.

### What the possible outcomes mean

**A. F1 improves and FPR remains controlled**
→ We have a genuine performance-improvement result. This becomes a strong candidate for the final NetShield method.

**B. PR-AUC/ROC-AUC remain strong but F1 does not improve**
→ The representation/domain adaptation is useful, but posterior calibration remains a bottleneck. The next method should target stronger unsupervised calibration/domain adaptation rather than adding arbitrary features.

**C. Estimated target prior is implausible or EM is unstable**
→ The prior-shift assumption is not adequate for this transfer. We should not force the correction.

### Important limitation
The EM correction assumes that the class-conditional score distributions are sufficiently stable across domains. Because UNSW and CICIDS contain different attack families and traffic-generation processes, this assumption may fail. The experiment is therefore a **controlled hypothesis test**, not a guaranteed solution.


In [ ]:
# Reproducibility manifest
manifest = {
    "phase": 15,
    "random_state": RANDOM_STATE,
    "representation": "expanded semantic (37 features)",
    "adaptation": "CORAL using unlabeled CICIDS adaptation sample",
    "classifier": {
        "type": "RandomForestClassifier",
        **RF_PARAMS,
        "class_weight": "balanced"
    },
    "calibration": "source-validation logistic/Platt-style score calibration",
    "prior_estimation": "unlabeled target EM prior-shift correction",
    "target_split": "Phase 12 fixed stratified 80/20 split, random_state=42",
    "adaptation_max": 300000,
    "phase12_f1_benchmark": PHASE12_BENCHMARK_F1,
    "target_labels_used_for_tuning": False
}

with open(TABLE_DIR / "04_experiment_config.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved Phase 15 outputs to:", TABLE_DIR)
